# CSRNet v2 — Pretrained, Combined-Density Training (ShanghaiTech Part A + Part B)

**Goal:** train a new CSRNet for both relatively sparse and very dense crowds, then compare it fairly against the existing Part-B checkpoint.

## Important experimental rules
- Your original checkpoint is **read-only**. This notebook never overwrites it.
- The new checkpoint is saved separately as `checkpoints/csrnet_v2_pretrained_AB_best.pth`.
- Training and validation are drawn from the official `train_data` folders. The official `test_data` folders are held out for final evaluation.
- Part A and Part B results are reported **separately** as well as together. A combined score alone can hide a failure on one density regime.
- We report MAE, RMSE, median relative error and the fraction of images within 10% / 20% error. The old notebook's clipped "accuracy" is not a standard accuracy metric.
- This is a reproducible experiment, not a promise of a particular accuracy. Performance depends on dataset integrity, labels, training time and camera-domain similarity.

## RTX 2050 (4 GB) settings
The default uses batch size 1 and random 512×512 crops with automatic mixed precision. If CUDA runs out of memory, reduce `CROP_SIZE` to 384 or 320. Do not change the model architecture or target-count scaling to solve an out-of-memory error.

## 1. Imports and reproducibility

In [1]:
import os, glob, time, math, random, copy, warnings, json, hashlib
from pathlib import Path
from collections import defaultdict

import cv2
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.io import loadmat
from scipy.spatial import cKDTree

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Reproducibility is preferred over maximum throughput for this experiment.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

PyTorch: 2.6.0+cu124
Device: cuda
GPU: NVIDIA GeForce RTX 2050
VRAM (GB): 4.0


## 2. Paths and training configuration
Change `DATA_ROOT` if the `ShanghaiTech` folder is not beside this notebook. Confirm the original checkpoint path before running training.

In [2]:
PROJECT_DIR = Path.cwd()
DATA_ROOT = PROJECT_DIR / "ShanghaiTech"

PART_A_TRAIN = DATA_ROOT / "part_A" / "train_data"
PART_A_TEST  = DATA_ROOT / "part_A" / "test_data"
PART_B_TRAIN = DATA_ROOT / "part_B" / "train_data"
PART_B_TEST  = DATA_ROOT / "part_B" / "test_data"

# Existing checkpoint is only loaded for baseline evaluation.
OLD_CHECKPOINT = PROJECT_DIR / "checkpoints" / "best_model_part_B.pth"

# New checkpoint is intentionally a different filename.
NEW_CHECKPOINT = PROJECT_DIR / "checkpoints" / "csrnet_v2_pretrained_AB_best.pth"
DENSITY_CACHE = PROJECT_DIR / ".csrnet_v2_density_cache"

# RTX 2050 4GB defaults
CROP_SIZE = 512       # try 384 or 320 only if CUDA OOM occurs
BATCH_SIZE = 1
NUM_WORKERS = 0       # reliable in Windows / VS Code notebooks
MAX_EPOCHS = 80
EARLY_STOPPING_PATIENCE = 12
VAL_FRACTION = 0.10
FRONTEND_LR = 1e-6
BACKEND_LR = 1e-4
WEIGHT_DECAY = 1e-4
USE_AMP = (device.type == "cuda")

# Pretrained VGG16 weights require a one-time download unless already cached.
USE_IMAGENET_PRETRAINED = True

print("Project:", PROJECT_DIR)
for name, p in [
    ("Part A train", PART_A_TRAIN), ("Part A test", PART_A_TEST),
    ("Part B train", PART_B_TRAIN), ("Part B test", PART_B_TEST),
    ("Old checkpoint", OLD_CHECKPOINT)
]:
    print(f"{name:16s}: {p} | exists={p.exists()}")

Project: d:\Mayank Raj\CSE NITD\5th_Semester_2026\CSBB 311 - Machine Learning\Project-ENDSEM
Part A train    : d:\Mayank Raj\CSE NITD\5th_Semester_2026\CSBB 311 - Machine Learning\Project-ENDSEM\ShanghaiTech\part_A\train_data | exists=True
Part A test     : d:\Mayank Raj\CSE NITD\5th_Semester_2026\CSBB 311 - Machine Learning\Project-ENDSEM\ShanghaiTech\part_A\test_data | exists=True
Part B train    : d:\Mayank Raj\CSE NITD\5th_Semester_2026\CSBB 311 - Machine Learning\Project-ENDSEM\ShanghaiTech\part_B\train_data | exists=True
Part B test     : d:\Mayank Raj\CSE NITD\5th_Semester_2026\CSBB 311 - Machine Learning\Project-ENDSEM\ShanghaiTech\part_B\test_data | exists=True
Old checkpoint  : d:\Mayank Raj\CSE NITD\5th_Semester_2026\CSBB 311 - Machine Learning\Project-ENDSEM\checkpoints\best_model_part_B.pth | exists=True


## 3. Check dataset layout before training
Expected layout: `part_A/train_data/images`, `part_A/train_data/ground-truth`, and equivalent folders for Part B and test data.

In [3]:
def image_files(split_dir):
    folder = Path(split_dir) / "images"
    if not folder.exists():
        return []
    paths = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        paths.extend(folder.glob(ext))
    return sorted(str(p) for p in paths)

for label, folder in [
    ("A train", PART_A_TRAIN), ("A test", PART_A_TEST),
    ("B train", PART_B_TRAIN), ("B test", PART_B_TEST)
]:
    files = image_files(folder)
    print(f"{label:8s}: {len(files):4d} images | folder exists={Path(folder).exists()}")

assert PART_A_TRAIN.exists() and PART_B_TRAIN.exists(), (
    "Part A and Part B train_data folders must both exist. Check DATA_ROOT."
)
assert len(image_files(PART_A_TRAIN)) > 0 and len(image_files(PART_B_TRAIN)) > 0, (
    "No train images found. Expected images/ subfolders."
)

A train :  300 images | folder exists=True
A test  :  182 images | folder exists=True
B train :  400 images | folder exists=True
B test  :  316 images | folder exists=True


## 4. CSRNet architecture

The architecture is kept compatible with the CSRNet definition in your original notebook (`VGG16 features[:23]` + six dilated backend convolutions + 1×1 output). The difference is that the new experiment initializes the VGG16 frontend with ImageNet weights.

The old checkpoint can therefore be loaded into the same architecture for baseline comparison.

In [4]:
class CSRNet(nn.Module):
    def __init__(self, pretrained_frontend=False):
        super().__init__()
        try:
            weights = models.VGG16_Weights.DEFAULT if pretrained_frontend else None
            vgg = models.vgg16(weights=weights)
        except AttributeError:
            vgg = models.vgg16(pretrained=pretrained_frontend)

        self.frontend = nn.Sequential(*list(vgg.features.children())[:23])
        self.backend = nn.Sequential(
            nn.Conv2d(512, 512, 3, dilation=2, padding=2), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, dilation=2, padding=2), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, dilation=2, padding=2), nn.ReLU(inplace=True),
            nn.Conv2d(512, 256, 3, dilation=2, padding=2), nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, dilation=2, padding=2), nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, 3, dilation=2, padding=2), nn.ReLU(inplace=True),
        )
        self.output_layer = nn.Conv2d(64, 1, kernel_size=1)
        self._initialize_backend_weights()

    def _initialize_backend_weights(self):
        for layer in self.backend.modules():
            if isinstance(layer, nn.Conv2d):
                nn.init.normal_(layer.weight, std=0.01)
                if layer.bias is not None:
                    nn.init.constant_(layer.bias, 0)
        nn.init.normal_(self.output_layer.weight, std=0.01)
        if self.output_layer.bias is not None:
            nn.init.constant_(self.output_layer.bias, 0)

    def forward(self, x):
        return self.output_layer(self.backend(self.frontend(x)))

print("CSRNet definition ready.")

CSRNet definition ready.


## 5. Ground-truth annotation and density-map generation

This experiment regenerates density maps consistently from ShanghaiTech point annotations (`GT_IMG_*.mat`) when available, rather than silently mixing density-map conventions from previous runs.

Each annotated head contributes approximately **one unit of density**. Gaussian kernels use a geometry-adaptive sigma based on nearby annotated heads, clipped to a practical range. Each truncated kernel is normalized over its visible patch, so border points also preserve their count.

If a `.mat` annotation is unavailable, an existing `.h5` density map is used as a fallback and its integral is checked. The notebook caches generated maps under `.csrnet_v2_density_cache` so they do not need to be regenerated every epoch.

In [6]:
def get_mat_points(mat_path):
    mat = loadmat(str(mat_path))
    if "image_info" not in mat:
        raise KeyError(f"'image_info' missing in {mat_path}")
    # ShanghaiTech annotation structure used by the original project.
    points = mat["image_info"][0, 0][0, 0][0]
    points = np.asarray(points, dtype=np.float32)
    if points.size == 0:
        return np.zeros((0, 2), dtype=np.float32)
    return points.reshape(-1, 2)

def generate_adaptive_density(height, width, points, beta=0.30, sigma_min=1.0, sigma_max=15.0):
    density = np.zeros((height, width), dtype=np.float32)
    points = np.asarray(points, dtype=np.float32).reshape(-1, 2)
    if len(points) == 0:
        return density

    if len(points) > 1:
        k = min(4, len(points))
        distances, _ = cKDTree(points).query(points, k=k)
        if distances.ndim == 1:
            distances = distances[:, None]
    else:
        distances = np.zeros((1, 1), dtype=np.float32)

    for idx, (x_f, y_f) in enumerate(points):
        x, y = int(round(float(x_f))), int(round(float(y_f)))
        if x < 0 or x >= width or y < 0 or y >= height:
            continue

        if len(points) > 1 and distances.shape[1] > 1:
            local = distances[idx, 1:]
            local = local[np.isfinite(local)]
            sigma = beta * float(local.mean()) if len(local) else 4.0
        else:
            sigma = 4.0
        sigma = float(np.clip(sigma, sigma_min, sigma_max))

        radius = max(1, int(math.ceil(3.0 * sigma)))
        x1, x2 = max(0, x-radius), min(width, x+radius+1)
        y1, y2 = max(0, y-radius), min(height, y+radius+1)

        xs = np.arange(x1, x2, dtype=np.float32)
        ys = np.arange(y1, y2, dtype=np.float32)
        xx, yy = np.meshgrid(xs, ys)
        kernel = np.exp(-((xx-x)**2 + (yy-y)**2) / (2.0*sigma*sigma)).astype(np.float32)
        kernel_sum = float(kernel.sum())
        if kernel_sum > 0:
            density[y1:y2, x1:x2] += kernel / kernel_sum

    return density

def annotation_path_for_image(image_path):
    p = Path(image_path)
    gt_dir = p.parent.parent / "ground-truth"
    return gt_dir / f"GT_{p.stem}.mat"

def load_or_create_density(image_path, image_size, cache_root=DENSITY_CACHE):
    image_path = Path(image_path)
    width, height = image_size
    part_name = "part_A" if "part_A" in str(image_path).replace("\\", "/") else "part_B"
    cache_dir = Path(cache_root) / part_name
    cache_dir.mkdir(parents=True, exist_ok=True)
    path_key = hashlib.sha1(str(image_path.resolve()).encode("utf-8")).hexdigest()[:12]
    cache_path = cache_dir / f"{image_path.parent.parent.name}_{image_path.stem}_{path_key}_{width}x{height}.npy"

    if cache_path.exists():
        density = np.load(cache_path)
        if density.shape == (height, width):
            return density.astype(np.float32), "cache"

    mat_path = annotation_path_for_image(image_path)
    if mat_path.exists():
        points = get_mat_points(mat_path)
        density = generate_adaptive_density(height, width, points)
        source = "MAT points (adaptive Gaussian)"
    else:
        h5_path = image_path.with_suffix(".h5")
        if not h5_path.exists():
            raise FileNotFoundError(f"No annotation .mat or density .h5 found for {image_path}")
        with h5py.File(h5_path, "r") as hf:
            key = "density" if "density" in hf else list(hf.keys())[0]
            density = np.asarray(hf[key], dtype=np.float32)
        if density.shape != (height, width):
            raise ValueError(f"H5 density shape {density.shape} != image {(height, width)}")
        source = "existing H5 density (fallback)"

    np.save(cache_path, density.astype(np.float32))
    return density.astype(np.float32), source

# Sanity check one image from each dataset.
for label, folder in [("Part A", PART_A_TRAIN), ("Part B", PART_B_TRAIN)]:
    p = image_files(folder)[0]
    with Image.open(p) as im:
        w, h = im.size
    d, src = load_or_create_density(p, (w, h))
    print(label, Path(p).name, "source:", src, "annotation count:", round(float(d.sum()), 3),
          "density shape:", d.shape)

Part A IMG_1.jpg source: MAT points (adaptive Gaussian) annotation count: 1546.0 density shape: (768, 1024)
Part B IMG_1.jpg source: MAT points (adaptive Gaussian) annotation count: 233.0 density shape: (768, 1024)


## 6. Deterministic train/validation split

The official `test_data` folders are not used for model selection. We split each official training folder into train and validation subsets with a fixed seed. This gives both density regimes a validation set.

The split is saved to `csrnet_v2_split.json` so the same files can be reused in later runs.

In [7]:
def split_paths(paths, val_fraction=VAL_FRACTION, seed=SEED):
    paths = list(paths)
    rng = random.Random(seed)
    rng.shuffle(paths)
    n_val = max(1, int(round(len(paths) * val_fraction)))
    val = sorted(paths[:n_val])
    train = sorted(paths[n_val:])
    return train, val

a_train_all = image_files(PART_A_TRAIN)
b_train_all = image_files(PART_B_TRAIN)
a_train, a_val = split_paths(a_train_all, seed=SEED + 1)
b_train, b_val = split_paths(b_train_all, seed=SEED + 2)

split_manifest = {
    "seed": SEED,
    "part_A_train": a_train, "part_A_val": a_val,
    "part_B_train": b_train, "part_B_val": b_val
}
with open(PROJECT_DIR / "csrnet_v2_split.json", "w", encoding="utf-8") as f:
    json.dump(split_manifest, f, indent=2)

print("Part A:", len(a_train), "train /", len(a_val), "validation")
print("Part B:", len(b_train), "train /", len(b_val), "validation")
print("Official test images remain separate.")

Part A: 270 train / 30 validation
Part B: 360 train / 40 validation
Official test images remain separate.


## 7. Dataset: balanced Part A + Part B training

Training uses random 512×512 crops (crop coordinates are multiples of 8) to reduce VRAM use on the RTX 2050. Horizontal flips are applied consistently to image and density map.

For every image/crop, the density map is resized to the exact CSRNet output size using area interpolation. It is then scaled by the pixel-area ratio so its **sum remains the head count**. This avoids assuming all image dimensions are divisible by eight.

In [8]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def image_to_tensor(rgb):
    x = rgb.astype(np.float32) / 255.0
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(x.transpose(2, 0, 1).copy()).float()

def density_to_output_grid(density, out_h, out_w):
    h, w = density.shape
    resized = cv2.resize(density, (out_w, out_h), interpolation=cv2.INTER_AREA)
    # Preserve the integral despite floor-sized VGG output dimensions.
    resized = resized * (float(h * w) / float(out_h * out_w))
    return resized.astype(np.float32)

class ShanghaiCrowdDataset(Dataset):
    def __init__(self, records, training=False, crop_size=CROP_SIZE):
        self.records = list(records)  # tuples: (image_path, domain)
        self.training = training
        self.crop_size = crop_size

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        image_path, domain = self.records[index]
        rgb = np.asarray(Image.open(image_path).convert("RGB"))
        h, w = rgb.shape[:2]
        density, _ = load_or_create_density(image_path, (w, h))

        if self.training:
            ch = min(self.crop_size, h)
            cw = min(self.crop_size, w)
            # Keep crop origin divisible by 8 so output/density alignment is exact.
            max_y = max(0, h - ch)
            max_x = max(0, w - cw)
            y_choices = list(range(0, max_y + 1, 8)) or [0]
            x_choices = list(range(0, max_x + 1, 8)) or [0]
            y = random.choice(y_choices)
            x = random.choice(x_choices)
            rgb = rgb[y:y+ch, x:x+cw]
            density = density[y:y+ch, x:x+cw]

            if random.random() < 0.5:
                rgb = np.ascontiguousarray(rgb[:, ::-1])
                density = np.ascontiguousarray(density[:, ::-1])

        ih, iw = rgb.shape[:2]
        out_h, out_w = ih // 8, iw // 8
        if out_h < 1 or out_w < 1:
            raise ValueError(f"Image too small for CSRNet: {image_path} {rgb.shape}")
        target = density_to_output_grid(density, out_h, out_w)

        return image_to_tensor(rgb), torch.from_numpy(target[None, ...]).float(), image_path, domain

train_records = [(p, "A") for p in a_train] + [(p, "B") for p in b_train]
val_records_A = [(p, "A") for p in a_val]
val_records_B = [(p, "B") for p in b_val]

train_dataset = ShanghaiCrowdDataset(train_records, training=True)
val_dataset_A = ShanghaiCrowdDataset(val_records_A, training=False)
val_dataset_B = ShanghaiCrowdDataset(val_records_B, training=False)

# Balance sampling probability between Part A and Part B.
domain_counts = {"A": len(a_train), "B": len(b_train)}
sample_weights = [1.0 / domain_counts[domain] for _, domain in train_records]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_records), replacement=True)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)
val_loader_A = DataLoader(val_dataset_A, batch_size=1, shuffle=False, num_workers=0)
val_loader_B = DataLoader(val_dataset_B, batch_size=1, shuffle=False, num_workers=0)

print("Train samples:", len(train_dataset))
print("Validation A/B:", len(val_dataset_A), "/", len(val_dataset_B))

Train samples: 630
Validation A/B: 30 / 40


## 8. Verify count conservation and tensor alignment before training
The density-map integral should approximately equal the annotation count. Also confirm that CSRNet output and target dimensions match.

In [9]:
# Inspect one sample from each domain.
for label, dataset in [("A", val_dataset_A), ("B", val_dataset_B)]:
    if len(dataset) == 0:
        continue
    x, y, path, domain = dataset[0]
    with torch.inference_mode():
        temp_model = CSRNet(pretrained_frontend=False).to(device).eval()
        out = temp_model(x.unsqueeze(0).to(device))
        del temp_model
    print(label, Path(path).name,
          "| input:", tuple(x.shape),
          "| target:", tuple(y.shape),
          "| model output:", tuple(out.shape),
          "| target count:", round(float(y.sum()), 3))
    assert tuple(y.shape[-2:]) == tuple(out.shape[-2:]), "Target/output shape mismatch"
    del out
    if device.type == "cuda":
        torch.cuda.empty_cache()

A IMG_100.jpg | input: (3, 654, 1024) | target: (1, 81, 128) | model output: (1, 1, 81, 128) | target count: 371.0
B IMG_108.jpg | input: (3, 768, 1024) | target: (1, 96, 128) | model output: (1, 1, 96, 128) | target count: 209.0


## 9. Build the new model with ImageNet-pretrained VGG16

This is a **new model**. The old checkpoint is not loaded into it. The pretrained VGG16 frontend is initialized from ImageNet; CSRNet backend/output layers are initialized separately.

If the pretrained-weight download fails, connect to the internet and rerun this cell. Do not silently set `pretrained_frontend=False` and call that the same experiment.

In [10]:
if USE_IMAGENET_PRETRAINED:
    model = CSRNet(pretrained_frontend=True).to(device)
else:
    raise RuntimeError("This experiment requires ImageNet-pretrained VGG16. Set USE_IMAGENET_PRETRAINED=True.")

print("New pretrained CSRNet initialized.")
print("New checkpoint destination:", NEW_CHECKPOINT)
print("Existing checkpoint will remain untouched:", OLD_CHECKPOINT)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\mayan/.cache\torch\hub\checkpoints\vgg16-397923af.pth
100%|██████████| 528M/528M [00:35<00:00, 15.5MB/s] 


New pretrained CSRNet initialized.
New checkpoint destination: d:\Mayank Raj\CSE NITD\5th_Semester_2026\CSBB 311 - Machine Learning\Project-ENDSEM\checkpoints\csrnet_v2_pretrained_AB_best.pth
Existing checkpoint will remain untouched: d:\Mayank Raj\CSE NITD\5th_Semester_2026\CSBB 311 - Machine Learning\Project-ENDSEM\checkpoints\best_model_part_B.pth


## 10. Validation metrics

We select checkpoints using the **macro mean relative absolute error**: compute relative absolute error per image, then average Part A and Part B domain means equally. This prevents the larger-count Part A samples from dominating model selection.

We also report Part A MAE/RMSE and Part B MAE/RMSE separately. These are the primary interpretable counting metrics.

In [11]:
@torch.inference_mode()
def evaluate_loader(model, loader, domain_name):
    model.eval()
    rows = []
    for images, targets, paths, domains in loader:
        images = images.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred = model(images)
        raw_count = float(pred.float().sum().item())
        pred_count = max(0.0, raw_count)
        gt_count = float(targets.sum().item())
        error = abs(pred_count - gt_count)
        rel_error = error / max(gt_count, 1.0)
        rows.append({
            "domain": domain_name,
            "image": Path(paths[0]).name,
            "path": paths[0],
            "ground_truth": gt_count,
            "prediction": pred_count,
            "raw_prediction": raw_count,
            "absolute_error": error,
            "relative_error": rel_error,
        })
    return pd.DataFrame(rows)

def summarize_metrics(df, label):
    if df.empty:
        return {"domain": label, "n": 0}
    e = df["prediction"].to_numpy() - df["ground_truth"].to_numpy()
    ae = np.abs(e)
    re = ae / np.maximum(df["ground_truth"].to_numpy(), 1.0)
    return {
        "domain": label,
        "n": len(df),
        "MAE": float(ae.mean()),
        "RMSE": float(np.sqrt(np.mean(e**2))),
        "Mean_relative_error_%": float(re.mean() * 100),
        "Median_relative_error_%": float(np.median(re) * 100),
        "Within_10%_%": float(np.mean(re <= 0.10) * 100),
        "Within_20%_%": float(np.mean(re <= 0.20) * 100),
    }

@torch.inference_mode()
def evaluate_both_domains(model, loader_A, loader_B):
    df_a = evaluate_loader(model, loader_A, "Part A")
    df_b = evaluate_loader(model, loader_B, "Part B")
    summaries = [
        summarize_metrics(df_a, "Part A"),
        summarize_metrics(df_b, "Part B"),
    ]
    rel_a = df_a["relative_error"].mean() if len(df_a) else np.nan
    rel_b = df_b["relative_error"].mean() if len(df_b) else np.nan
    macro_rel = float(np.nanmean([rel_a, rel_b]))
    return df_a, df_b, pd.DataFrame(summaries), macro_rel

## 11. Training loop: mixed precision, separate learning rates, early stopping

- Frontend learning rate: `1e-6` (small updates to ImageNet features).
- Backend/output learning rate: `1e-4`.
- Loss: summed pixelwise MSE on the density maps.
- AMP: enabled on CUDA to reduce memory use.
- Gradient clipping: 10.
- Scheduler: ReduceLROnPlateau on macro validation relative error.
- Early stopping: 12 epochs without improvement.

The best new model is saved only to `NEW_CHECKPOINT`.

In [12]:
def save_checkpoint(path, model, optimizer, scheduler, epoch, best_score, config):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_macro_relative_error": best_score,
        "config": config,
    }, path)

def train_model(model, train_loader, val_loader_A, val_loader_B):
    frontend_params = list(model.frontend.parameters())
    other_params = list(model.backend.parameters()) + list(model.output_layer.parameters())

    optimizer = torch.optim.Adam(
        [
            {"params": frontend_params, "lr": FRONTEND_LR},
            {"params": other_params, "lr": BACKEND_LR},
        ],
        weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=4, min_lr=1e-7
    )
    criterion = nn.MSELoss(reduction="sum")
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_score = float("inf")
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        losses = []
        start = time.time()

        for step, (images, targets, _, _) in enumerate(train_loader, start=1):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                outputs = model(images)
                loss = criterion(outputs, targets)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().item()))

            if step % 25 == 0 or step == len(train_loader):
                print(
                    f"Epoch {epoch:03d}/{MAX_EPOCHS} | "
                    f"step {step:04d}/{len(train_loader)} | "
                    f"loss {losses[-1]:.2f}",
                    end="\r"
                )

        # Validate on the held-out subsets from train_data, not official test_data.
        df_a, df_b, summary, macro_rel = evaluate_both_domains(
            model, val_loader_A, val_loader_B
        )
        scheduler.step(macro_rel)

        row = {
            "epoch": epoch,
            "train_loss_mean": float(np.mean(losses)),
            "val_macro_relative_error": macro_rel,
            "val_A_MAE": float(summary.loc[summary.domain == "Part A", "MAE"].iloc[0]),
            "val_B_MAE": float(summary.loc[summary.domain == "Part B", "MAE"].iloc[0]),
            "lr_frontend": optimizer.param_groups[0]["lr"],
            "lr_backend": optimizer.param_groups[1]["lr"],
            "epoch_seconds": time.time() - start,
        }
        history.append(row)
        print("\n", row)

        if macro_rel < best_score:
            best_score = macro_rel
            epochs_without_improvement = 0
            save_checkpoint(
                NEW_CHECKPOINT, model, optimizer, scheduler,
                epoch, best_score,
                {
                    "seed": SEED, "crop_size": CROP_SIZE,
                    "frontend_lr": FRONTEND_LR, "backend_lr": BACKEND_LR,
                    "train_domains": ["Part A", "Part B"],
                    "density_target": "adaptive Gaussian from ShanghaiTech points",
                }
            )
            print(f"Saved new best checkpoint: {NEW_CHECKPOINT}")
        else:
            epochs_without_improvement += 1
            print(f"No improvement for {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE} epochs.")

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping.")
            break

        if device.type == "cuda":
            torch.cuda.empty_cache()

    return pd.DataFrame(history)

RUN_TRAINING = True  # Set True only after all checks above pass.
print("Set RUN_TRAINING=True to start the new training experiment.")

Set RUN_TRAINING=True to start the new training experiment.


## 12. Start training

Training can take a substantial amount of time on an RTX 2050. It is not realistic to promise that 80 epochs will finish in a few minutes. Start with the defaults, monitor GPU memory and validation metrics, and allow early stopping to decide when to stop.

If you receive CUDA out-of-memory:
1. Set `CROP_SIZE = 384` (or 320).
2. Recreate the dataset/loaders by rerunning Sections 7 onward.
3. Keep batch size at 1 and AMP enabled.

In [ ]:
if RUN_TRAINING:
    history_df = train_model(model, train_loader, val_loader_A, val_loader_B)
    display(history_df.tail(10))
    if not history_df.empty:
        fig, ax = plt.subplots(1, 2, figsize=(13, 4))
        ax[0].plot(history_df["epoch"], history_df["train_loss_mean"])
        ax[0].set_title("Training summed MSE")
        ax[0].set_xlabel("Epoch")
        ax[1].plot(history_df["epoch"], history_df["val_macro_relative_error"] * 100)
        ax[1].set_title("Validation macro relative error (%)")
        ax[1].set_xlabel("Epoch")
        plt.tight_layout()
        plt.show()
else:
    print("Training not started. Set RUN_TRAINING=True in Section 11, then rerun this cell.")

C:\Users\mayan\AppData\Local\Temp\ipykernel_23916\3673163418.py:28: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
C:\Users\mayan\AppData\Local\Temp\ipykernel_23916\3673163418.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


C:\Users\mayan\AppData\Local\Temp\ipykernel_23916\166516406.py:7: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):



 {'epoch': 1, 'train_loss_mean': 45.854663327576255, 'val_macro_relative_error': 0.3893586955431779, 'val_A_MAE': 244.51345876057943, 'val_B_MAE': 51.84504880905151, 'lr_frontend': 1e-06, 'lr_backend': 0.0001, 'epoch_seconds': 241.53015637397766}
Saved new best checkpoint: d:\Mayank Raj\CSE NITD\5th_Semester_2026\CSBB 311 - Machine Learning\Project-ENDSEM\checkpoints\csrnet_v2_pretrained_AB_best.pth


C:\Users\mayan\AppData\Local\Temp\ipykernel_23916\3673163418.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


C:\Users\mayan\AppData\Local\Temp\ipykernel_23916\166516406.py:7: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):



 {'epoch': 2, 'train_loss_mean': 38.301488476593256, 'val_macro_relative_error': 0.23976560222729204, 'val_A_MAE': 179.391343943278, 'val_B_MAE': 32.4561740398407, 'lr_frontend': 1e-06, 'lr_backend': 0.0001, 'epoch_seconds': 229.6706428527832}
Saved new best checkpoint: d:\Mayank Raj\CSE NITD\5th_Semester_2026\CSBB 311 - Machine Learning\Project-ENDSEM\checkpoints\csrnet_v2_pretrained_AB_best.pth


... (12 outputs left out)

C:\Users\mayan\AppData\Local\Temp\ipykernel_23916\3673163418.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


C:\Users\mayan\AppData\Local\Temp\ipykernel_23916\166516406.py:7: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):



 {'epoch': 7, 'train_loss_mean': 33.654933433276454, 'val_macro_relative_error': 0.14146892330204608, 'val_A_MAE': 112.19526011149088, 'val_B_MAE': 16.796473598480226, 'lr_frontend': 1e-06, 'lr_backend': 0.0001, 'epoch_seconds': 151.96347856521606}
No improvement for 2/12 epochs.


C:\Users\mayan\AppData\Local\Temp\ipykernel_23916\3673163418.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


## 13. Load best new checkpoint

Always evaluate the best checkpoint, not simply the final epoch weights.

In [ ]:
def load_model_checkpoint(model, path):
    obj = torch.load(path, map_location=device)
    state = obj["model_state_dict"] if isinstance(obj, dict) and "model_state_dict" in obj else obj
    if any(k.startswith("module.") for k in state):
        state = {k.removeprefix("module."): v for k, v in state.items()}
    model.load_state_dict(state, strict=True)
    model.eval()
    return obj

if NEW_CHECKPOINT.exists():
    new_model = CSRNet(pretrained_frontend=False).to(device)
    new_checkpoint_info = load_model_checkpoint(new_model, NEW_CHECKPOINT)
    print("Loaded new checkpoint:", NEW_CHECKPOINT)
    if isinstance(new_checkpoint_info, dict):
        print("Best epoch:", new_checkpoint_info.get("epoch"))
        print("Best validation macro relative error:",
              new_checkpoint_info.get("best_macro_relative_error"))
else:
    new_model = None
    print("New checkpoint not found. Train first.")

## 14. Load the old model without modifying its checkpoint

This section supports the original notebook's plain `state_dict` format and common wrapped checkpoint formats. If the old checkpoint cannot be loaded strictly, the notebook reports the error rather than partially loading mismatched weights.

In [ ]:
def extract_state_dict(obj):
    if isinstance(obj, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
        if obj and all(torch.is_tensor(v) for v in obj.values()):
            return obj
    raise ValueError("No recognizable state_dict found.")

old_model = None
if OLD_CHECKPOINT.exists():
    try:
        old_model = CSRNet(pretrained_frontend=False).to(device)
        old_obj = torch.load(OLD_CHECKPOINT, map_location=device)
        old_state = extract_state_dict(old_obj)
        if any(k.startswith("module.") for k in old_state):
            old_state = {k.removeprefix("module."): v for k, v in old_state.items()}
        old_model.load_state_dict(old_state, strict=True)
        old_model.eval()
        print("Old checkpoint loaded for baseline:", OLD_CHECKPOINT)
    except Exception as exc:
        old_model = None
        print("Could not load old checkpoint strictly:", repr(exc))
else:
    print("Old checkpoint not found at:", OLD_CHECKPOINT)

## 15. Final test-set evaluation: old vs new, Part A and Part B

The official test folders are evaluated only here. Do not tune the model or calibration based on these results. The old and new models receive the same RGB images and ImageNet normalization.

If the old model was trained only on Part B, its Part A performance is expected to be weaker. The new model's purpose is to improve both domains, so inspect both rows rather than relying on one combined number.

In [ ]:
def make_eval_loader(test_dir):
    records = [(p, "A" if "part_A" in str(p).replace("\\", "/") else "B")
               for p in image_files(test_dir)]
    ds = ShanghaiCrowdDataset(records, training=False)
    return DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)

test_loader_A = make_eval_loader(PART_A_TEST)
test_loader_B = make_eval_loader(PART_B_TEST)
print("Official test sizes:", len(test_loader_A), "Part A |", len(test_loader_B), "Part B")

def compare_models_on_test(old_model, new_model, loader_A, loader_B):
    result_frames = {}
    summary_rows = []

    for model_name, candidate in [("Old CSRNet", old_model), ("New CSRNet v2", new_model)]:
        if candidate is None:
            continue
        for domain, loader in [("Part A", loader_A), ("Part B", loader_B)]:
            df = evaluate_loader(candidate, loader, domain)
            df["model"] = model_name
            result_frames[(model_name, domain)] = df
            summary = summarize_metrics(df, domain)
            summary["model"] = model_name
            summary_rows.append(summary)

    summary_df = pd.DataFrame(summary_rows)
    if not summary_df.empty:
        cols = ["model", "domain", "n", "MAE", "RMSE",
                "Mean_relative_error_%", "Median_relative_error_%",
                "Within_10%_%", "Within_20%_%"]
        summary_df = summary_df[[c for c in cols if c in summary_df.columns]]
    return summary_df, result_frames

if new_model is not None:
    comparison_summary, comparison_details = compare_models_on_test(
        old_model, new_model, test_loader_A, test_loader_B
    )
    display(comparison_summary)
else:
    comparison_summary, comparison_details = None, {}
    print("Train and load the new model before final comparison.")

## 16. Save reproducible test results
The CSV files include per-image predictions and errors for later comparison/reporting.

In [ ]:
if comparison_summary is not None:
    summary_path = PROJECT_DIR / "csrnet_v2_test_summary.csv"
    comparison_summary.to_csv(summary_path, index=False)
    print("Saved:", summary_path)

    for (model_name, domain), frame in comparison_details.items():
        safe_name = model_name.lower().replace(" ", "_")
        safe_domain = domain.lower().replace(" ", "_")
        path = PROJECT_DIR / f"{safe_name}_{safe_domain}_predictions.csv"
        frame.to_csv(path, index=False)
        print("Saved:", path)

## 17. Inspect worst errors for each model/domain

This helps distinguish systematic overcounting, undercounting, and domain-specific failures. A high count error can remain even when the heatmap looks visually plausible.

In [ ]:
for key, frame in comparison_details.items():
    print("\n", key)
    display(frame.sort_values("absolute_error", ascending=False).head(10))

## 18. Visual comparison: input, ground-truth density, old prediction, new prediction

The heatmaps below are independently normalized for display. Their colors are **not** confidence values; counts come from the unnormalized density-map integral.

In [ ]:
@torch.inference_mode()
def get_density_prediction(candidate_model, image_path):
    rgb = np.asarray(Image.open(image_path).convert("RGB"))
    x = image_to_tensor(rgb).unsqueeze(0).to(device)
    with torch.cuda.amp.autocast(enabled=USE_AMP):
        pred = candidate_model(x)
    return rgb, pred[0, 0].float().cpu().numpy()

def display_model_comparison(image_path, old_model, new_model):
    rgb = np.asarray(Image.open(image_path).convert("RGB"))
    density_gt, _ = load_or_create_density(image_path, (rgb.shape[1], rgb.shape[0]))
    pred_maps = {}
    for label, candidate in [("Old", old_model), ("New", new_model)]:
        if candidate is not None:
            _, pred_maps[label] = get_density_prediction(candidate, image_path)

    h, w = rgb.shape[:2]
    maps = [("Ground truth", density_gt)] + list(pred_maps.items())
    fig, axes = plt.subplots(1, 1 + len(maps), figsize=(5 * (1 + len(maps)), 4))
    axes[0].imshow(rgb)
    axes[0].set_title(Path(image_path).name)
    axes[0].axis("off")

    for ax, (label, dmap) in zip(axes[1:], maps):
        count = float(dmap.sum())
        vis = cv2.resize(dmap, (w, h), interpolation=cv2.INTER_LINEAR)
        im = ax.imshow(vis, cmap="jet")
        ax.set_title(f"{label}\nCount: {count:.1f}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

# Choose an official test image from each domain.
if len(image_files(PART_A_TEST)) and len(image_files(PART_B_TEST)):
    display_model_comparison(image_files(PART_A_TEST)[0], old_model, new_model)
    display_model_comparison(image_files(PART_B_TEST)[0], old_model, new_model)

## 19. Test a real-world photograph

Set the path below. For a non-ShanghaiTech photograph, no actual error/accuracy can be computed unless you have a reliable manually counted ground truth or point annotations. The prediction alone is not evidence of accuracy.

In [ ]:
CUSTOM_IMAGE = ""  # Example: r"D:\CrowdProject\your_photo.jpg"

if CUSTOM_IMAGE and Path(CUSTOM_IMAGE).exists():
    for label, candidate in [("Old CSRNet", old_model), ("New CSRNet v2", new_model)]:
        if candidate is None:
            continue
        rgb, density_map = get_density_prediction(candidate, CUSTOM_IMAGE)
        pred_count = max(0.0, float(density_map.sum()))
        print(label, "| predicted people:", round(pred_count, 2),
              "| raw density integral:", round(float(density_map.sum()), 2))

        h, w = rgb.shape[:2]
        heat = cv2.resize(density_map, (w, h), interpolation=cv2.INTER_LINEAR)
        heat = np.maximum(heat, 0)
        heat_u8 = cv2.normalize(heat, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        heat_rgb = cv2.cvtColor(cv2.applyColorMap(heat_u8, cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted(rgb, 0.6, heat_rgb, 0.4, 0)

        fig, ax = plt.subplots(1, 3, figsize=(16, 4))
        ax[0].imshow(rgb); ax[0].set_title("Input")
        ax[1].imshow(heat_rgb); ax[1].set_title("Display-normalized density")
        ax[2].imshow(overlay); ax[2].set_title(f"{label}: {pred_count:.1f}")
        for a in ax: a.axis("off")
        plt.tight_layout(); plt.show()
else:
    print("Set CUSTOM_IMAGE to a valid image path to test a real-world photograph.")

## 20. Optional: continue training from the new best checkpoint

This is for resuming an interrupted v2 run only. It is not for loading the old Part-B checkpoint. The saved checkpoint contains optimizer and scheduler states; the simple training function above starts a fresh optimizer if rerun. For a fully interrupted/resumed run, load those states explicitly before continuing.

## 21. What success should look like

Do not target a made-up "85% accuracy" number. Compare:
- Part A MAE/RMSE against the old model.
- Part B MAE/RMSE against the old model.
- Within-10% and within-20% rates.
- Error distributions and worst-case images.

A useful outcome is a meaningful improvement on both test domains without a severe regression in either. If Part A improves while Part B collapses (or vice versa), the combined training has not solved the two-domain objective.

## 22. Limitations and next step for real CCTV
ShanghaiTech Part A/B are benchmark datasets, not a guarantee of performance on a historical photo or your target CCTV camera. For deployment, collect representative footage from the intended camera positions, annotate a held-out set, and evaluate it separately. Keep safety thresholds and any actuator actions disabled until site-specific validation is complete.